# 00 - Project Overview and Repo Audit

## Goal
The project keeps monthly FRED macro data, engineered macro features, fixed split conventions, baseline models, and uncertainty tooling in transparent notebooks.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.project_utils import ensure_project_dirs, seed_everything

seed_everything(42)
paths = ensure_project_dirs()

print("Repository root:", paths["root"])
for key in ["raw_data", "processed_data", "models", "results"]:
    print(f"{key}: {paths[key]}")

Repository root: C:\Users\User\Downloads\261-Project
raw_data: C:\Users\User\Downloads\261-Project\data\raw
processed_data: C:\Users\User\Downloads\261-Project\data\processed
models: C:\Users\User\Downloads\261-Project\models
results: C:\Users\User\Downloads\261-Project\results


## Project Baseline (Current Setup)

This project uses a clear monthly macro setup:

- Data source: monthly FRED macro series.
- Series set: `CPI`, `Unemployment`, `InterestRate`, `M2`, `Treasury10Y`, `OilPrice`, `PPI`, `RetailSales`, `Sentiment`, `Employment`, `HousingStarts`.
- Engineered feature family: first differences, returns, YoY transforms, squared terms, interactions, rolling inflation stats, lagged inflation, seasonality terms, and a COVID dummy.
- Default data split logic: chronological 70% train, 15% validation, 15% test with context windows for validation/test sequence construction.
- Lag defaults: `60` months (main flow) and `36` months (one rolling-backtest mode).
- Models: stacked LSTM plus XGBoost baseline.
- Metrics: MAE, sMAPE, MASE.
- Extras: rolling backtest, permutation importance, MC-dropout uncertainty, conformal prediction intervals, and output artifacts under `results/`.

The notebook flow keeps these assumptions explicit and reproducible under `data/`, `models/`, and `results/`.

## Target Definition Decision

A common source of confusion in inflation projects is mixing two target definitions:

- year-over-year CPI inflation,
- month-over-month CPI percent change.

### Final design decision
This project supports **both target definitions** through one config cell in the data/feature notebooks:

- `mom_pct`: month-over-month CPI percent change.
- `yoy_pct`: year-over-year CPI percent change.

Default is **`mom_pct`** in this notebook, and later notebooks keep the target name explicit in code and outputs to avoid ambiguity.

In [2]:
TARGET_DEFINITION = "mom_pct"  # options: "mom_pct", "yoy_pct"
FORECAST_HORIZON_MONTHS = 1
DEFAULT_N_LAGS = 60
BACKTEST_N_LAGS = 36
SPLIT_RATIOS = (0.70, 0.15, 0.15)
RANDOM_SEED = 42

TARGET_OPTIONS = {
    "mom_pct": "Monthly CPI percent change (default).",
    "yoy_pct": "Year-over-year CPI percent change.",
}

if TARGET_DEFINITION not in TARGET_OPTIONS:
    raise ValueError(f"Unsupported target definition: {TARGET_DEFINITION}")

print("Target definition:", TARGET_DEFINITION)
print("Description:", TARGET_OPTIONS[TARGET_DEFINITION])
print("Forecast horizon (months):", FORECAST_HORIZON_MONTHS)
print("Split ratios (train/val/test):", SPLIT_RATIOS)
print("Lag defaults:", {"main": DEFAULT_N_LAGS, "backtest": BACKTEST_N_LAGS})

Target definition: mom_pct
Description: Monthly CPI percent change (default).
Forecast horizon (months): 1
Split ratios (train/val/test): (0.7, 0.15, 0.15)
Lag defaults: {'main': 60, 'backtest': 36}


## Planned notebook sequence

1. `00_project_overview_and_repo_audit.ipynb` (this notebook)
2. `01_data_ingestion_and_target_definition.ipynb`
3. `02_feature_engineering_and_splits.ipynb`
4. `03_classical_baselines.ipynb`
5. `04_lstm_nowcasting.ipynb`
6. `05_xgboost_and_tree_models.ipynb`
7. `06_backtesting_and_uncertainty.ipynb`
8. `07_ablation_and_model_selection.ipynb`

Each notebook runs top-to-bottom on a fresh clone after `pip install -r requirements.txt`, using either `FRED_API_KEY` or local cached files.

In [3]:
import pandas as pd

series_list = [
    "CPI",
    "Unemployment",
    "InterestRate",
    "M2",
    "Treasury10Y",
    "OilPrice",
    "PPI",
    "RetailSales",
    "Sentiment",
    "Employment",
    "HousingStarts",
]

engineered_features = [
    "Unemp_d", "Rate_d", "Oil_ret", "PPI_yoy", "M2_yoy", "Retail_yoy",
    "Employment_d", "Housing_d", "T10Y", "Sentiment", "PPI_yoy_sq", "Oil_ret_sq",
    "Rate_d_sq", "Unemp_d_sq", "Rate_Unemp", "Oil_PPI", "Infl_ma3", "Infl_ma6",
    "Infl_ema", "Infl_vol6", "Inflation_prev", "MoY_sin", "MoY_cos", "COVID"
]

summary = pd.DataFrame(
    {
        "item": ["raw_series_count", "engineered_feature_candidates"],
        "value": [len(series_list), len(engineered_features)],
    }
)

print("Raw FRED series:", ", ".join(series_list))
summary

Raw FRED series: CPI, Unemployment, InterestRate, M2, Treasury10Y, OilPrice, PPI, RetailSales, Sentiment, Employment, HousingStarts


,item,value
0,raw_series_count,11
1,engineered_feature_candidates,24
